# Cloud-Assisted Inference Security Evaluation — Reproducibility Record

**Dissertation:** *Machine Learning on the Edge: Performance and Security Evaluation of Artificial Neural Network Implementations in Embedded Systems*  
**Results:** Chapter IV, Section 4.7.3 — Cloud-Assisted Inference Security Results  
**Repository branch:** `Cloud-Security`  
**Evidence review:** November 5, 2026

This notebook records how security-relevant configurations and implementation details were reviewed for the cloud-assisted FPGA, PC, and Meta Quest 3S experiments. It preserves evidence provenance, observed settings, potential implications, and the boundaries of the qualitative review. It is **not** a penetration-testing record, a performance results notebook, or an assertion that identified attack paths were exploited.

The seven newly collected screenshots are included in this branch. Cloud deployment source code, service documentation, and performance benchmarks remain in their original repository branches.

## 1. Evidence Inventory and Branch Boundaries

The following **seven screenshots** are the new image files for `Cloud-Security`; place them in the same directory as this notebook to render the images below.

| File | Evidence |
|---|---|
| `wifi_settings.png` | PC's `USAResearch` Wi-Fi properties (WPA2-Personal, Public profile, DHCP) |
| `open_settings.png` | Open Wi-Fi properties observed in Windows; user confirmed that Quest 3S used the university's `OPEN` network |
| `ethernet_properties.png` | Windows Ethernet properties on the university network; **not** ZedBoard adapter properties |
| `cloud_networking.png` | Cloud Run ingress, HTTPS endpoints, VPC and optional networking controls |
| `cloud_log.png` | Representative Cloud Run request logs and Gunicorn process messages |
| `cloud_iam_permissions.png` | Project IAM role assignments for runtime identity and administrator |
| `cloud_containers.png` | Cloud Run container resources, startup probe, and displayed integration options |

**Already maintained in the `Cloud` branch (do not duplicate):** `Cloud_MNIST_Security.png` (Cloud Run Security settings), application source, dependency declarations, and other Cloud Run evidence. See [the Cloud branch](https://github.com/kjstacey/J00287507-Dissertation-AI-on-the-Edge/tree/Cloud) for the source artifacts. The service YAML and Cloud Run settings were also inspected during the review; this notebook documents relevant observed values without treating a typed transcription as an attached screenshot.

**Important:** Evidence captured on November 5, 2026 documents the reviewed configuration; where earlier deployment state is not independently timestamped, the exact historical setting is not independently proven.

## 2. Review Method and Reproduction Sequence

1. Review dissertation methodology for the cloud-assisted FPGA and Unity deployments, including data flow and TLS implementation.
2. Inspect each client's network medium and local-network configuration.
3. Inspect the Cloud Run **Security**, **Networking**, **Containers**, and **YAML** configurations.
4. Match the Cloud Run runtime service-account name in YAML with its IAM role assignment.
5. Review the Python Flask `/predict` handler and `requirements.txt` in the `Cloud` branch.
6. Inspect representative Cloud Run operational logs and identify fields actually present.
7. Separate **direct observations**, **code-supported conclusions**, **potential security implications**, and **unverified controls**.

The evaluation was qualitative. No packet interception, penetration test, unauthorized invocation, IAM escalation, ransomware execution, or resource-exhaustion test was performed. Do not modify current infrastructure merely to reproduce historical evidence.

## 3. Network Paths and Scope

| Client | Local network path | Remote request path |
|---|---|---|
| ZedBoard FPGA | University Ethernet through the office telephone's data passthrough to the wall connection | lwIP + Mbed TLS 2.28.x, TCP/443, TLS 1.2, HTTPS POST to Cloud Run |
| HP ZBook | University `USAResearch` Wi-Fi | UnityWebRequest over HTTPS to Cloud Run |
| Meta Quest 3S | University `OPEN` Wi-Fi | UnityWebRequest over HTTPS to Cloud Run |

The three clients used the university's internet connectivity, but **the physical routing, switches, wireless access points, VLANs, and upstream security controls were not independently mapped**.

The Quest remained connected to `OPEN` Wi-Fi during both local and cloud-assisted experiments. Its **local classification did not require sending inference requests to Cloud Run**, despite the active Wi-Fi connection.

Link-layer wireless protection and end-to-end transport protection must be evaluated independently. A wired Ethernet link is not proof of encrypted data; an open Wi-Fi link is not proof that HTTPS application payloads traveled unencrypted.

## 4. PC — Research Wi-Fi Configuration

![PC research Wi-Fi properties](wifi_settings.png)

**Observed in Windows:**

- SSID `USAResearch`; Wi-Fi 6 (802.11ax), 5 GHz; security type **WPA2-Personal**.
- **Public** Windows network profile, which should be interpreted alongside the existing PC-security firewall evidence.
- IP address and DNS settings assigned automatically through DHCP.
- Random hardware addresses off; DNS entries shown as **unencrypted**.
- The research wireless password was described by the researcher as shared with individuals affiliated with university research activities. Password strength was **not measured**.

**Interpretation:** WPA2-Personal protected the local wireless link but did not identify individual users through enterprise wireless credentials. The relevant application transport used HTTPS independently. The screenshot does not establish the access point's backend configuration, connected-user list, or exact security state during every experiment.

## 5. Quest 3S — Open Wireless Network

![Open Wi-Fi properties captured on Windows](open_settings.png)

**Evidence boundary:** This is a **Windows screenshot of the university `OPEN` network**, not a headset Wi-Fi settings screenshot. The researcher confirmed that the experimental Quest 3S used the university's `OPEN` wireless connection throughout local and cloud-assisted testing.

- SSID `OPEN` displayed with **Security type: Open**.
- A WPA2/WPA3 wireless-link protection mechanism was not shown.
- The Unity cloud client used HTTPS through UnityWebRequest.
- The screenshot alone does not establish Quest-specific Wi-Fi adapter settings, DNS settings, or the precise wireless access point in use.

**Interpretation:** The open wireless link lacked the WPA2-Personal protection present on `USAResearch`; transport-level HTTPS remained a separate protection for application requests. No wireless capture or TLS-certificate-substitution test was performed.

## 6. ZedBoard — Wired Network Connection

![Windows Ethernet network properties](ethernet_properties.png)

The researcher reported that the **ZedBoard's Ethernet port** connected to the data-output passthrough on the office telephone, which connected to the university wall Ethernet outlet.

The included screenshot documents **the HP laptop's Windows Ethernet interface**, with a Public profile, DHCP configuration, Intel I219-LM adapter, and 100/100 Mbps reported link speed. **These adapter and speed details must not be attributed to the ZedBoard.**

For the FPGA, methodology and the reviewed embedded application establish:

- ZedBoard processing-system networking using **lwIP**.
- **Mbed TLS 2.28.x**, TLS 1.2 over port 443.
- Cipher suite `TLS-ECDHE-ECDSA-WITH-AES-128-GCM-SHA256`.
- **Server certificate verification disabled** and **deterministic development entropy**.

The wired route reduced reliance on a radio link, but did not remove network interception or server impersonation risks. TLS encrypted traffic, yet the disabled certificate verification prevented normal validation of the remote endpoint. The ZedBoard's own assigned IP/DNS, switch-port configuration, and campus network access controls were not separately inspected.

## 7. Cloud Run — Networking and Ingress

![Cloud Run networking](cloud_networking.png)

**Observed service:** `mnist-cloud-inference`, `us-east1`.

| Setting | Observed |
|---|---|
| Ingress | **All**; direct access from the internet |
| Default HTTPS endpoint | Enabled, Google-managed `run.app` endpoints |
| Custom domain | None shown |
| Optional SSH connection | Not enabled |
| Outbound VPC connection | Not configured |
| End-to-end HTTP/2 | Not enabled |
| Session affinity | Not enabled |

**Interpretation:** Public ingress was allowed. The absence of optional SSH or end-to-end HTTP/2 is not itself evidence of an exploitable vulnerability. No conclusion about a private VPC, outbound isolation, or application authorization follows solely from this screenshot.

## 8. Cloud Run — Authentication, Encryption and Threat Detection

The **existing** [`Cloud_MNIST_Security.png` evidence in the Cloud branch](https://github.com/kjstacey/J00287507-Dissertation-AI-on-the-Edge/blob/Cloud/Cloud_MNIST_Security.png) displays:

- **Require authentication** selected and IAM checked.
- An explicit warning that `allUsers` has service permissions, so the endpoint **is publicly accessible**.
- Identity-Aware Proxy (IAP) unchecked; custom audiences absent.
- **Google-managed encryption key** selected, rather than customer-managed Cloud KMS.
- The displayed threat-detection feature **disabled**.

**Critical interpretation:** The selected “Require authentication” UI state did **not** ensure that each client was authenticated: the `allUsers` binding permitted unauthenticated invocation. Invocation permissions and runtime service-account permissions are distinct. Google-managed encryption keys refer to cloud-managed encryption at rest and do not prove application payloads were never logged or saved.

## 9. Cloud Run — Container Configuration

![Cloud Run container settings](cloud_containers.png)

**Observed:**

- Image stored under Google Artifact Registry `cloud-run-source-deploy/mnist-cloud-inference`; container port **8080**.
- **1 vCPU** and **512 MiB** memory per instance.
- Startup TCP probe on port 8080, timeout **240 seconds**; startup CPU boost enabled.
- No environment variables, Secret Manager references, mounted volumes, or Cloud SQL connections **displayed** in this screenshot.
- Extra ephemeral disk not enabled; execution environment set to Default.

These are deployment and operational observations. They do **not** prove absence of secrets in code, filesystem writes, other cloud resource access, or supply-chain vulnerabilities. A startup probe detects service availability, not client authenticity.

## 10. Cloud Run YAML — Revision, Runtime Identity and Limits

The reviewed Cloud Run service YAML reported:

| Field | Recorded value |
|---|---|
| `metadata.creationTimestamp` | `2026-10-15T20:49:16.139677Z` |
| `run.googleapis.com/ingress` | `all` |
| Runtime identity (`serviceAccountName`) | `221036165238-compute@developer.gserviceaccount.com` |
| Runtime identity type | Compute Engine default service account |
| `containerConcurrency` | **80** per instance |
| `autoscaling.knative.dev/maxScale` | **3** instances |
| `timeoutSeconds` | **300** |
| Container limits | `cpu: 1000m`, `memory: 512Mi` |
| `run.googleapis.com/build-enable-automatic-updates` | `false` |
| `latestReadyRevisionName` | `mnist-cloud-inference-00002-plx` |
| Traffic routing | 100% to latest revision |
| Artifact image | An Artifact Registry image pinned by a SHA-256 digest |

The digest was observed in the YAML, providing an identifier for the image that was deployed. The digest does **not** independently establish image signing, secure provenance, scanning, or model-file integrity.

**Availability distinction:** Instance caps, concurrency and timeouts bound service operation, but they are **not** an application-level per-client rate limit. No resource-exhaustion test was conducted.

## 11. Project IAM — Runtime Permissions

![Cloud IAM project permissions](cloud_iam_permissions.png)

The Cloud Run runtime identity from YAML matched the principal shown in the IAM screenshot:

| Principal | Displayed role | Finding |
|---|---|---|
| Compute Engine default service account | **Editor (Legacy)**, project-level | Broad permissions beyond the minimal inference task |
| Project administrator | **Owner (Legacy)** | Administrative project privileges |

The default runtime identity's project-level Editor role did not implement least privilege for a service whose documented inference task consisted of request processing and classification.

**Do not conflate:** `allUsers` invocation permission does **not** give an anonymous caller the Editor role. A *separate compromise* of the running workload or misuse of its credentials would be required for that runtime privilege to become relevant to an attacker. No such compromise was observed or tested.

IAM screenshot records are evidence of displayed grants, not a complete review of inherited policies, organization restrictions, or every possible effective permission.

## 12. Python Service and Model-Loading Review

The application source maintained in the [`Cloud` branch](https://github.com/kjstacey/J00287507-Dissertation-AI-on-the-Edge/tree/Cloud) implements Flask routes `/` and `POST /predict`.

**Reviewed behavior:**

- Loads the model at startup from `WeigntsAndBiases.txt` through `network2.load(...)`.
- Reads request body with `request.get_data()` and rejects lengths other than **784 bytes** with HTTP 400.
- Converts `uint8` input to floating-point, divides by 128, and reshapes to `(784, 1)`.
- Calls `net.feedforward(x)` and returns only a predicted digit in JSON.
- Contains **no application-level API key/token verification**, **no per-client request throttling**, and **no cryptographic signature/hash validation** of the model in the reviewed script.
- Does not explicitly write submitted image bytes or prediction values to permanent storage or logs.

**Limitations:** The body is read **before** length checking, and the reviewed script does not define its own maximum request-body size. This observation alone does not establish practical exploitability because gateway and runtime limits may apply. The exact source/build artifact used for historical runs was not independently attested against the current text review.

## 13. Python Dependencies and Container Build

The supplied `requirements.txt` declared:

| Package | Version | Purpose |
|---|---|---|
| Flask | `3.0.3` | Python request handler |
| Gunicorn | `23.0.0` | Application server |
| NumPy | `1.26.4` | Numerical input conversion |

The service YAML indicated a Cloud Run source deployment and identified the resulting Artifact Registry image. A project-authored Dockerfile was **not established**; a source-based build does not necessarily require one.

The pinned top-level dependency list does not identify every transitive dependency or OS package in the image. No image vulnerability-scan report, dependency lock file, signature, or exploit test was presented for the deployment. Distinguish a possible advisories-based risk from a vulnerability confirmed in the running configuration.

## 14. Cloud Run Operational Logs

![Cloud Run request logs](cloud_log.png)

A representative October 28, 2026 log view showed:

- Repeated `POST` requests to `/predict` returning **HTTP 200**.
- Timestamps, status, displayed response size of **133 B**, and service-reported durations mostly **2–3 ms**.
- A truncated client identifier beginning `UnityPlay...`, consistent with Unity-client requests.
- Later Gunicorn log messages such as `Handling signal: term`, `Worker exiting`, and `Shutting down: Master`.

**Logging conclusion:** The **displayed records** contain request metadata, not image bytes or predicted digits. The reviewed Flask code contains no explicit payload logging. Neither observation proves that no payload data could be retained elsewhere in the platform. The Gunicorn termination messages by themselves do not demonstrate a crash or attack.

The displayed service-side request duration is not interchangeable with the dissertation's end-to-end client request-latency measurements.

## 15. Data Transmission and Protection Boundaries

| Boundary | Observed | Security qualification |
|---|---|---|
| Sensor/image preprocessing to client request | One prepared MNIST image represented as 784 bytes | Smaller than a raw camera frame, but still meaningful image data |
| Client to cloud | HTTPS POST to `/predict` | Transport protection depends on client-side TLS implementation |
| Server processing | Binary input decoded and classified by locally loaded model | Endpoint validates length, not sender identity or input authenticity |
| Cloud to client | JSON predicted digit | No application-layer message authentication identified beyond HTTPS |
| Cloud observability | Request metadata in displayed logs | Full platform retention behavior not established |

The model was located in the cloud container rather than transferred with each request. No evidence of per-request model download or explicit permanent storage of submitted images appeared in the reviewed application.

The FPGA's disabled server certificate verification is an especially relevant trust-boundary limitation. The PC and Quest used operating-environment HTTPS networking; their underlying validation behavior was not independently probed.

## 16. Consolidated Security Findings

| Area | Verified observation | Resulting concern / boundary |
|---|---|---|
| FPGA TLS | TLS 1.2; certificate verification off; deterministic entropy | Encrypted connection without verified server identity; nonproduction entropy |
| PC Wi-Fi | WPA2-Personal shared research credential | Shared-key network admission, without individual Wi-Fi user authentication |
| Quest Wi-Fi | University `OPEN` network | No WPA2/WPA3 wireless-link protection; HTTPS remains separate |
| Cloud ingress | `All` | Publicly reachable service |
| Invocation | `allUsers` permitted | Unauthenticated requests possible |
| Runtime permissions | Default Compute Engine service account with Editor (Legacy) | Privileges broader than needed for inference |
| Model protection | Cloud model file loaded without in-code hash/signature verification | Model integrity depends on deployment and upstream controls not tested |
| Input checks | Length must equal 784 bytes | Inadequate to authenticate caller or input content |
| Availability | Max 3 instances, concurrency 80, timeout 300 s | Bounds capacity; no per-client rate limit in application |
| Logging | Request metadata shown; no payload in displayed entries | Complete retention and observability settings not verified |

No single security score was calculated. No inference that an attack occurred should be drawn from a potential weakness alone.

## 17. Evidence and Applicability Limitations

- Screenshots reflect a **November 5, 2026** review unless an earlier record is explicitly dated. The operational-log excerpt includes **October 28, 2026** requests.
- `open_settings.png` and `ethernet_properties.png` are **Windows** property screenshots; they are contextual evidence, not direct screenshots of the Quest and ZedBoard network interfaces.
- Network SSIDs, wired connection arrangement, and Quest connectivity across experimental modes were also documented through the researcher's direct observations.
- Campus network segmentation, access-point policy, firewall rules, DHCP and DNS infrastructure, and physical network topology were not independently verified.
- Certificate verification details were directly identified for FPGA; OS-managed TLS behavior for Unity clients was not independently subjected to interception testing.
- Source review does not attest to every OS, Python, Gunicorn, or transitive dependency inside the running image.
- Google-managed encryption at rest does not by itself establish payload retention, key-access permissions, or application-level confidentiality.
- Security controls were **not selectively enabled or disabled** to measure separate effects on latency, throughput, energy, or other performance metrics.
- No security attacks, malicious requests, unauthorized cloud access, credential misuse, or deliberate service interruption were carried out.

The dissertation's later security/performance trade-off analysis is intentionally deferred until the remaining datasets are completed.

## 18. Reproducibility Checklist

- [ ] Review the seven screenshot files by their exact names in this branch.
- [ ] Verify the Windows `USAResearch` network is shown as WPA2-Personal and Public.
- [ ] Verify the `OPEN` network screenshot shows Security type Open and preserve its Windows-specific provenance.
- [ ] Verify the wired-interface screenshot pertains to Windows, not ZedBoard hardware.
- [ ] Confirm Cloud Run Networking shows `All` ingress and default HTTPS endpoint enabled.
- [ ] Review existing `Cloud_MNIST_Security.png` in the `Cloud` branch for the `allUsers` warning and Google-managed encryption.
- [ ] Confirm that YAML runtime service account matches the IAM screenshot's Editor (Legacy) principal.
- [ ] Inspect the supplied Flask source for exact-length validation, model loading, and missing application-level authentication.
- [ ] Compare the top-level requirements versions to the source deployment record.
- [ ] Confirm displayed Cloud Run logs contain request metadata but not image payloads.
- [ ] Keep unverified settings and untested attack scenarios explicitly qualified.

**Related branches:** `Cloud` (service implementation and existing security settings), cloud-assisted FPGA and Unity code/data branches, `FPGA-Security`, `PC-Security`, `Quest3s-security`, and `Manuals-and-Specifications` for reference documentation.